# 04b · Direction analysis — CPU only, no model

Everything numeric that `04` printed but did not save, plus the check that decides what the
direction actually is. Runs entirely on the cached activations from `04a`, so it takes seconds and
needs no GPU.

The question this notebook exists to settle: `04` fitted the direction on **experimenter ground
truth**, but the model's displayed answer agrees with ground truth on most kept items. So a
"displayed-answer" direction and a "ground-truth" direction are partly confounded, and steering
alone cannot separate them. The separation is available in the data:

- Split the kept set by the behavioural covariate in `data/keep_pairs.json`: items where the
  display **agrees** with the truth label, and items where it **contradicts** it.
- Fit and evaluate within each subgroup. If the direction encodes the true answer, it separates
  the classes in **both** subgroups with the same sign. If it encodes the answer the model is
  about to display, the projections **reverse sign** on the contradicting subgroup, because there
  the displayed answer is the negation of the label.
- Independently, fit a direction on **displayed-answer** labels and take the cosine with the
  ground-truth direction. Two names for one axis if it is near 1; genuinely different axes if not.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import json, csv, numpy as np, os
RUN   = os.environ.get("AEE_RUN", "run_4")
CACHE = f"/content/drive/MyDrive/aee/cache/{RUN}"
RES   = f"results/{RUN}"; os.makedirs(RES, exist_ok=True)

META  = json.load(open(f"{CACHE}/activations_pairs_meta.json"))
ACT   = np.load(f"{CACHE}/activations_pairs.npy").astype(np.float32)
items = json.load(open("data/extraction_pairs.json"))["questions"]
KS    = json.load(open("data/keep_pairs.json"))
assert META["ids"] == [it["id"] for it in items]

pairs = sorted({it["pair_id"] for it in items})
rng   = np.random.default_rng(0); rng.shuffle(pairs)
cut   = int(0.6*len(pairs)); FIT, TEST = set(pairs[:cut]), set(pairs[cut:])

KEEP_PAIRS = set(KS["keep_pairs"])
KEEP   = np.array([it["pair_id"] in KEEP_PAIRS for it in items])
IS_YES = np.array([it["answer"] == "yes" for it in items])
IN_FIT = np.array([it["pair_id"] in FIT for it in items])
IN_DOM = np.array([it["domain"] == "in_domain" for it in items])

# behavioural covariate: did the display contradict the truth label for THIS item?
inv_y = set(KS["display_inverted_yes_half"]); inv_n = set(KS["display_inverted_no_half"])
INV = np.array([(it["pair_id"] in inv_y) if it["answer"]=="yes" else (it["pair_id"] in inv_n)
                for it in items])
N_STATES = ACT.shape[1]
print(f"{ACT.shape} | kept {KEEP.sum()} | display contradicts label on {int((INV&KEEP).sum())} kept items")

In [ ]:
unit = lambda x: x/np.linalg.norm(x)
def direction(L, mask, labels):
    A = ACT[:, L, :]; return A[mask & labels].mean(0) - A[mask & ~labels].mean(0)
def cohens_d(a, b):
    na, nb = len(a), len(b)
    sp = np.sqrt(((na-1)*a.var(ddof=1) + (nb-1)*b.var(ddof=1))/(na+nb-2))
    return (a.mean()-b.mean())/sp
def score(L, fit_mask, test_mask, labels=None):
    """Direction AND threshold are fitted on fit_mask only. 04 set the threshold from the
    midpoint of the TEST class means, which leaks the test labels into the classifier and makes
    its accuracy optimistic; Cohen's d is a two-sample statistic and was unaffected."""
    labels = IS_YES if labels is None else labels
    v = direction(L, fit_mask, labels)
    if np.linalg.norm(v) < 1e-8: return None
    vh = unit(v)
    fy, fn = ACT[fit_mask & labels, L, :] @ vh, ACT[fit_mask & ~labels, L, :] @ vh
    thr = 0.5*(fy.mean() + fn.mean())                      # <- fit split
    py, pn = ACT[test_mask & labels, L, :] @ vh, ACT[test_mask & ~labels, L, :] @ vh
    thr_leak = 0.5*(py.mean() + pn.mean())                 # what 04 used, kept for comparison
    acc = lambda t: float(np.concatenate([py > t, pn <= t]).mean())
    return dict(d=float(cohens_d(py,pn)), acc=acc(thr), acc_leaky=acc(thr_leak),
                n=int((test_mask & labels).sum() + (test_mask & ~labels).sum()),
                norm=float(np.linalg.norm(v)), v=v)

FIT_M, TEST_M = IN_FIT & KEEP, ~IN_FIT & KEEP
rows = [(L, score(L, FIT_M, TEST_M)) for L in range(1, N_STATES)]
rows = [(L, r) for L, r in rows if r]
ranked = sorted(rows, key=lambda t: -abs(t[1]["d"]))
LAYER, BEST = ranked[0]
print("top 5:", "  ".join(f"L{L} d={r['d']:.3f} acc={r['acc']:.3f}" for L,r in ranked[:5]))
print(f"\nselected L{LAYER}: ||v||={BEST['norm']:.2f}  d={BEST['d']:.3f}  n_test={BEST['n']}")
print(f"  acc (threshold from fit split, honest) = {BEST['acc']:.3f}")
print(f"  acc (threshold from test means, as 04 reported) = {BEST['acc_leaky']:.3f}")
v_truth = BEST["v"]; v_hat = unit(v_truth)

## 1 · Robustness of the screen, and topic transfer

In [ ]:
S = {}
un = score(LAYER, IN_FIT, ~IN_FIT)
S["unscreened"] = dict(d=un["d"], acc=un["acc"], n=un["n"], cos_with_screened=float(v_hat @ unit(un["v"])))
print(f"unscreened (all 150): d={un['d']:.3f} acc={un['acc']:.3f}   cos(screened, unscreened)={S['unscreened']['cos_with_screened']:.4f}")

for name, f_, t_ in [("in->out", IN_DOM & KEEP, ~IN_DOM & KEEP), ("out->in", ~IN_DOM & KEEP, IN_DOM & KEEP)]:
    r = score(LAYER, f_, t_); S[name] = dict(d=r["d"], acc=r["acc"], n=r["n"])
    print(f"{name:9s} d={r['d']:6.3f} acc={r['acc']:.3f}  n={r['n']}")
v_in, v_out = unit(direction(LAYER, IN_DOM & KEEP, IS_YES)), unit(direction(LAYER, ~IN_DOM & KEEP, IS_YES))
S["cos_in_out"] = float(v_in @ v_out)
print(f"cos(v_in_domain, v_out_domain) = {S['cos_in_out']:.4f}   <- ~1 means one axis, not two topic axes")

## 2 · The decisive test — ground truth or displayed answer?

`AGREE` = display matched the truth label. `CONTRA` = display contradicted it. Both are scored with
**the same ground-truth labels** and the same direction fitted on the fit split.

- separates both subgroups, same sign -> the axis tracks the **true answer**
- separates `AGREE` and reverses on `CONTRA` -> the axis tracks the **answer about to be displayed**

In [ ]:
vh = v_hat
for name, m in [("AGREE  (display == label)", KEEP & ~INV), ("CONTRA (display != label)", KEEP & INV)]:
    py, pn = ACT[m & IS_YES, LAYER, :] @ vh, ACT[m & ~IS_YES, LAYER, :] @ vh
    if len(py) < 2 or len(pn) < 2: print(f"{name}: too few items ({len(py)}/{len(pn)})"); continue
    thr = 0.5*(py.mean()+pn.mean())
    d   = float(cohens_d(py, pn)); acc = float(np.concatenate([py>thr, pn<=thr]).mean())
    S[name.split()[0]] = dict(d=d, acc=acc, n_yes=len(py), n_no=len(pn))
    print(f"{name}: d={d:+.3f}  acc={acc:.3f}  (n {len(py)} yes / {len(pn)} no)")

# an independent axis fitted on displayed-answer labels
DISPLAYED = IS_YES ^ INV                      # what the model asserted
v_disp = direction(LAYER, KEEP & IN_FIT, DISPLAYED)
S["cos_truth_vs_displayed"] = float(v_hat @ unit(v_disp))
S["frac_labels_agree"] = float((DISPLAYED[KEEP] == IS_YES[KEEP]).mean())
print(f"\ncos(v_ground_truth, v_displayed_answer) = {S['cos_truth_vs_displayed']:.4f}"
      f"   (the two label sets agree on {S['frac_labels_agree']:.0%} of kept items)")

In [ ]:
S["selected_layer"] = LAYER; S["d_test"] = BEST["d"]; S["acc_test"] = BEST["acc"]
S["norm"] = BEST["norm"]; S["n_test"] = BEST["n"]; S["n_kept_prompts"] = int(KEEP.sum())
S["sweep"] = [dict(layer=L, norm=r["norm"], d=r["d"], acc=r["acc"], acc_leaky=r["acc_leaky"]) for L, r in rows]
json.dump(S, open(f"{RES}/truth_direction_summary.json", "w"), indent=1)
print("saved ->", f"{RES}/truth_direction_summary.json")